# VIS model 

In [1]:

from utils import get_device
from data_process_v4 import get_loaders, class_cols
from data_process_v4 import TASK_3_TEST_LABELS_DIR

from VIS_V1 import create_model, train_model
from utils import get_device
import torch
import pandas as pd
from tqdm import tqdm

Using CSV_PATH: data/Task_3/ISIC2018_Task3_Training_GroundTruth.csv
Using IMAGES_DIR: data/Task_3/Train_images
Using VAL CSV: data/Task_3/ISIC2018_Task3_Validation_GroundTruth.csv
Using VAL IMAGES_DIR: data/Task_3/Validation_images
Using TEST CSV: data/Task_3/ISIC2018_Task3_Test_GroundTruth.csv
Using TEST IMAGES_DIR: data/Task_3/Test_images


In [2]:
device = get_device()

loaders = get_loaders(
    image_size=(384, 384),
    num_workers=0,
    test_csv_path=TASK_3_TEST_LABELS_DIR,
)


train_loader, val_loader, test_loader, train_df, val_df, test_df = loaders
class_cols_order = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
num_classes = 7
model = create_model(num_classes=num_classes, model_name="vit_base_patch16_384")


class_counts = [int(train_df[c].sum()) for c in class_cols_order]
model_name = "vis_model.pht"
#base_model = create_model(num_classes=num_classes, model_name="vit_base_patch16_384")
#ckpt = torch.load(model_name, map_location=device)
#base_model.load_state_dict(ckpt["model_state"])


model, history2 = train_model(
    train_loader,
    val_loader,
    num_classes=num_classes,
    device=device,
    epochs=20,                 
    class_counts=class_counts,
    #model=base_model,          
    save_path=model_name
)


KeyboardInterrupt: 

# resnet

In [ ]:
import torch
from resnet_v2 import train_model, create_resnet_model
from eval_resnet import evaluate_model

In [ ]:
loaders = get_loaders(
    image_size=(650, 400),                 
    num_workers=0,
    test_csv_path=TASK_3_TEST_LABELS_DIR,
)

train_loader, val_loader, test_loader, train_df, val_df, test_df = loaders

class_cols_order = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
num_classes = 7

# class distribution for BalancedFocalLoss
class_counts = [int(train_df[c].sum()) for c in class_cols_order]

model = create_resnet_model(
     num_classes=num_classes,
     pretrained=True,
     backbone_name="resnet50",
     head_hidden_dim=512,
     head_dropout=0.3,
 )

model, history = train_model(
    train_loader=train_loader,
    val_loader=val_loader,
    num_classes=num_classes,
    epochs=10,
    lr=3e-4,
    weight_decay=0.05,
    class_counts=class_counts,              
    save_path="best_resnet50_resizer.pth",
    device=device,                          
    backbone_name="resnet50",
    head_hidden_dim=512,
    head_dropout=0.3,
)


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

In [ ]:

device = get_device()
print("Using device:", device)

class_cols_order = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]

# evaluate on test set, compute mean recall and plot confusion matrix
mean_recall, cm, (y_true, y_pred) = evaluate_model(
    model=model,
    data_loader=test_loader,
    device=device,
    class_names=class_cols_order,   # used as axis labels in confusion matrix
    num_classes=len(class_cols_order),
    verbose=True,
)

print("Mean recall (macro):", mean_recall)